# Delegating with `super()`: 13 new advanced problems, taught step by step

**A second, tutorial-style practice notebook** | Python 3.10+ | Standard library only

This is a **new set of problems**, not a reprint of the earlier problem-and-solution notebook. Its structure deliberately follows the uploaded lesson: introduce one idea in a markdown cell, show a very small piece of code, inspect behavior, explain what just happened, and only then extend the example. There are frequent *predict → run → explain → test* cycles rather than one huge answer per problem.

**Main subject:** inheritance, method delegation, `super()`, the original instance, initialization, properties, and dynamic dispatch. **Clearly labeled extensions:** cooperative MRO, exception safety, `__slots__`, factories, immutable objects, context managers, asynchronous methods, abstract methods, and an integrated serialization design.

Every problem contains its requirements, intermediate reasoning, a fully worked implementation, test cases, and a concluding design lesson. Run the notebook **top to bottom**. It makes no network calls, creates no external files, and requires no third-party packages.

## How to use this tutorial

1. When you reach a **Prediction**, write down the expected result before executing the next code cell.
2. Read the **What happened?** explanation after running the example. Do not skip it: the important part is *why* Python took that path.
3. Complete the next step mentally, then inspect the provided working solution. Every solution has `assert`-based checks or a captured, intentional exception.
4. All class names carry a `P01_`, `P02_`, etc. prefix so demonstrations do not accidentally overwrite one another when run sequentially.

**Important correction to a tempting oversimplification:** `super()` is **not** a synonym for “call my immediate parent.” For instance methods it creates a proxy that continues attribute lookup **after a particular class in the MRO** while binding the found method to the original object. In a simple single-inheritance chain this commonly reaches a parent implementation; in multiple inheritance it may reach a sibling branch.

## Orientation: a tiny instrument for investigating lookup

We will occasionally print `type(obj).__mro__`. This is the actual method resolution order for the instance's class, not just an inheritance diagram. For methods found through `super()`, `self` is still the same instance. An overridden method's calls to `self.other_method()` always perform normal lookup from the instance's type, a fact that will matter in several challenges.

In [1]:
def show_mro(cls):
    """Display the actual Python MRO in a compact form."""
    print(" -> ".join(base.__name__ for base in cls.__mro__))

print("Setup complete: no third-party dependencies.")

Setup complete: no third-party dependencies.


## Problem 01 — Why `super()` can call a sibling branch

**Level:** Advanced · diagnostic  
**Your assignment:** Predict the exact call order of a cooperative multiple-inheritance hierarchy and explain why the next method need not belong to a direct parent.

We will build the solution in small pieces. Read each prediction question before running the following cell; then compare your prediction with the output and assertions.

### Step 1 — Start with an endpoint

The bottom class is deliberately boring: it records that it ran and returns a string. Later overrides must return the result of the next method rather than replacing it.

In [2]:
class P01_Core:
    def route(self):
        self.events.append("core")
        return "core"

### Step 2 — Add two independently reusable wrappers

Both wrappers use `super().route()`. Neither wrapper names the class that will come next. This is what makes them *cooperative*: another class can be inserted into the MRO without rewriting either wrapper.

In [3]:
class P01_Left(P01_Core):
    def route(self):
        self.events.append("left")
        return "L(" + super().route() + ")"


class P01_Right(P01_Core):
    def route(self):
        self.events.append("right")
        return "R(" + super().route() + ")"

### Step 3 — Make the diamond and predict

Create `P01_Combined(P01_Left, P01_Right)`. **Prediction:** Will `P01_Left.route()` delegate straight to `P01_Core.route()` because `P01_Core` is its declared base, or will `P01_Right.route()` run in between? Predict both the returned string and `events`.

In [4]:
class P01_Combined(P01_Left, P01_Right):
    def __init__(self):
        self.events = []


show_mro(P01_Combined)
combined = P01_Combined()
result = combined.route()
print("result:", result)
print("events:", combined.events)

P01_Combined -> P01_Left -> P01_Right -> P01_Core -> object
result: L(R(core))
events: ['left', 'right', 'core']


### What happened?

The combined MRO places `P01_Left` before `P01_Right` before `P01_Core`. Therefore `super()` **inside `P01_Left`**, when invoked on a combined instance, proceeds to `P01_Right`. Each wrapper contributes exactly once; the endpoint executes once. The nesting of the string reverses the visible return order relative to the forward call order.

In [5]:
assert result == "L(R(core))"
assert combined.events == ["left", "right", "core"]
assert P01_Combined.__mro__[:4] == (
    P01_Combined, P01_Left, P01_Right, P01_Core
)
print("P01 solved: sibling dispatch follows the instance class's MRO.")

P01 solved: sibling dispatch follows the instance class's MRO.


### Design lesson

`super()` is defined relative to both the **class where the call appears** and the **MRO of the receiver**. Directly calling `P01_Core.route(self)` from `P01_Left` would bypass the right wrapper. Prefer cooperative calls when independent mixins must all participate; use an explicit base-class call only when intentionally bypassing cooperation and you understand the consequences.

## Problem 02 — Dynamic dispatch and the hidden recursion trap

**Level:** Advanced · method semantics  
**Your assignment:** Trace an ancestor method that calls a child override through `self`, identify a recursion-prone design, and replace it with a proper extension hook.

We will build the solution in small pieces. Read each prediction question before running the following cell; then compare your prediction with the output and assertions.

### Step 1 — Define a stable base workflow

Our base workflow calls `self.prepare()`. That lookup is not frozen to the base class: `self` keeps its runtime type even when `run()` was reached through `super()`.

In [6]:
class P02_Base:
    def run(self):
        return "base-run:" + self.prepare()

    def prepare(self):
        return "base-prepare"

### Step 2 — Override both methods

**Prediction:** What does `P02_Child().run()` return? The child `run()` calls `super().run()`, which calls `self.prepare()`. Decide which class supplies that `prepare()` before executing.

In [7]:
class P02_Child(P02_Base):
    def run(self):
        return "child-run:" + super().run()

    def prepare(self):
        return "child-prepare"


value = P02_Child().run()
print(value)

child-run:base-run:child-prepare


### What happened?

The base's `run()` runs with a **child instance** as `self`. Consequently `self.prepare()` selects the child's override. `super()` changes where lookup starts for the particular delegated call; it does **not** turn `self` into an instance of the parent class.

In [8]:
assert value == "child-run:base-run:child-prepare"
assert isinstance(P02_Child(), P02_Base)

### Step 3 — Diagnose a tempting wrong fix without crashing the kernel

Suppose someone wrote `P02_Child.prepare()` as `return super().run()` in an attempt to reuse base logic. The call graph would be `child.run → base.run → child.prepare → base.run → child.prepare → ...`. This creates unbounded recursion, so we will **not** execute it. Correct approach: have the override of `prepare()` delegate to **`super().prepare()`**, the same operation it overrides.

In [9]:
class P02_Safe(P02_Base):
    def run(self):
        return "safe-run:" + super().run()

    def prepare(self):
        return "safe-" + super().prepare()


safe = P02_Safe()
print(safe.run())
assert safe.run() == "safe-run:base-run:safe-base-prepare"

safe-run:base-run:safe-base-prepare


### Design lesson

Draw the call graph when a parent calls a virtual method through `self`. In extensions, match an overriding method to its corresponding ancestor method (`prepare → prepare`) unless you deliberately want to re-enter a broader workflow. This avoids accidental recursion and preserves predictable hooks.

## Problem 03 — Preserve an invariant with a template method

**Level:** Advanced · API design  
**Your assignment:** Implement a validated public workflow that specialized subclasses extend through a hook, without copying validation or allowing an invalid state transition.

We will build the solution in small pieces. Read each prediction question before running the following cell; then compare your prediction with the output and assertions.

### Step 1 — Specify the invariant

A ledger's public `post(amount)` operation must reject non-finite, non-positive, and boolean values. A failed post must not change `balance` or `entries`. The base class owns validation; subclasses should extend **one narrow hook** rather than duplicate the whole public method.

In [10]:
import math
from numbers import Real


class P03_Ledger:
    def __init__(self):
        self.balance = 0.0
        self.entries = []

    def post(self, amount):
        if (isinstance(amount, bool) or not isinstance(amount, Real)
                or not math.isfinite(float(amount)) or amount <= 0):
            raise ValueError("amount must be a finite positive real number")
        self._record(float(amount))
        return self.balance

    def _record(self, amount):
        self.entries.append(amount)
        self.balance += amount

### Step 2 — Add behavior at the correct extension point

An audit subclass should record every successful posting. It uses `super().__init__()` to establish base state, then supplies its own `audit` list. Its `_record()` calls `super()._record()` **before** adding the audit event. Predict whether an invalid amount can ever reach this hook through the normal public API.

In [11]:
class P03_AuditedLedger(P03_Ledger):
    def __init__(self):
        super().__init__()
        self.audit = []

    def _record(self, amount):
        super()._record(amount)
        self.audit.append(("posted", amount, self.balance))


ledger = P03_AuditedLedger()
print("after first post:", ledger.post(12.5), ledger.audit)

after first post: 12.5 [('posted', 12.5, 12.5)]


### What happened?

`post()` inherited from the base performs validation, then dynamically dispatches to the subclass's `_record()`. That override delegates the actual balance and entry update to the parent, and augments it with an audit event. It is the **same ledger instance** throughout, so all mutations refer to one `balance` and one `entries` list.

### Step 3 — Test success and failure states

We explicitly capture the expected exception rather than leaving a notebook-stopping error cell. The before/after snapshot checks not just the exception type but also the key correctness condition: failure does not mutate state.

In [12]:
assert ledger.post(7.5) == 20.0
assert ledger.entries == [12.5, 7.5]
assert len(ledger.audit) == 2

snapshot = (ledger.balance, list(ledger.entries), list(ledger.audit))
for invalid in (-1, 0, float("nan"), float("inf"), True):
    try:
        ledger.post(invalid)
    except ValueError:
        pass
    else:
        raise AssertionError(f"Expected a ValueError for {invalid!r}")
    assert (ledger.balance, ledger.entries, ledger.audit) == snapshot
print("P03 solved: all invalid inputs leave state unchanged.")

P03 solved: all invalid inputs leave state unchanged.


### Design lesson

This is a template-method pattern: keep invariant checks in the public method and give subclasses a documented hook. The hook assumes callers respect the public API; it is not a security boundary against arbitrary direct calls to `_record()`. If subclasses must not modify validation, document and test that contract rather than relying on naming conventions alone.

## Problem 04 — Route keyword arguments through cooperative constructors

**Level:** Expert · multiple inheritance  
**Your assignment:** Create a four-class initialization chain with disjoint keyword-only contracts, prove that each initializer executes once, and detect unknown arguments instead of silently ignoring them.

We will build the solution in small pieces. Read each prediction question before running the following cell; then compare your prediction with the output and assertions.

### Step 1 — Write the strict endpoint

The terminal base consumes **no application-specific options**. It rejects leftovers so typos such as `naem=` cannot disappear into `**kwargs`. Each mixin will remove exactly the option that belongs to it before forwarding the rest.

In [13]:
class P04_Root:
    def __init__(self, **kwargs):
        if kwargs:
            raise TypeError(f"unknown initializer options: {sorted(kwargs)}")
        self.steps = ["root"]

### Step 2 — Add independent named and timed layers

Both layers inherit from `P04_Root`, yet when placed together their `super().__init__(**kwargs)` calls cooperate along the **combined object's MRO**. Each layer sets its own attribute only after downstream initialization succeeds.

In [14]:
class P04_Named(P04_Root):
    def __init__(self, *, name, **kwargs):
        super().__init__(**kwargs)
        if not isinstance(name, str) or not name.strip():
            raise ValueError("name must be nonempty")
        self.name = name
        self.steps.append("named")


class P04_Timed(P04_Root):
    def __init__(self, *, started_at, **kwargs):
        super().__init__(**kwargs)
        if not isinstance(started_at, int) or isinstance(started_at, bool):
            raise ValueError("started_at must be an integer")
        self.started_at = started_at
        self.steps.append("timed")

### Step 3 — Complete the combined class

**Prediction:** The MRO starts with `P04_Job`, then `P04_Named`, then `P04_Timed`, then `P04_Root`. Because our log statements happen *after* the delegated call, predict the **reverse** initialization-completion order in `steps`.

In [15]:
class P04_Job(P04_Named, P04_Timed):
    def __init__(self, *, job_id, **kwargs):
        super().__init__(**kwargs)
        if not isinstance(job_id, str) or not job_id:
            raise ValueError("job_id must be nonempty")
        self.job_id = job_id
        self.steps.append("job")


show_mro(P04_Job)
job = P04_Job(job_id="J-9", name="daily-import", started_at=123)
print(job.__dict__)

P04_Job -> P04_Named -> P04_Timed -> P04_Root -> object
{'steps': ['root', 'timed', 'named', 'job'], 'started_at': 123, 'name': 'daily-import', 'job_id': 'J-9'}


### What happened?

The calls go **down** the MRO as `Job → Named → Timed → Root`, while the post-delegation actions finish **up** as `Root → Timed → Named → Job`. No initializer calls a named direct parent, so there is no duplicate root call in this hierarchy. This is particularly useful for reusable mixins with independent configuration parameters.

### Step 4 — Make typo detection an explicit test

Unknown keywords should fail at the endpoint. In this exercise a typo must not produce a partially usable object; Python does not return the object to the caller if its constructor raises. The implementation does not promise rollback of external side effects in constructors.

In [16]:
assert job.steps == ["root", "timed", "named", "job"]
assert (job.name, job.started_at, job.job_id) == ("daily-import", 123, "J-9")
try:
    P04_Job(job_id="J-10", name="batch", started_at=0, naem="typo")
except TypeError as exc:
    assert "naem" in str(exc)
else:
    raise AssertionError("unknown options must be rejected")
print("P04 solved: cooperative kwargs and strict endpoint verified.")

P04 solved: cooperative kwargs and strict endpoint verified.


### Design lesson

For cooperative `__init__` designs: consume only your own arguments, forward remaining ones exactly once, and provide a terminal policy for unexpected arguments. Validate before delegation if validation must prevent any downstream side effects; our example validates fields after delegation to demonstrate completion order, not to claim atomic constructors.

## Problem 05 — Extend a method while preserving errors and cleanup

**Level:** Expert · exception safety  
**Your assignment:** Add logging and cleanup around a delegated operation, preserve its successful return value, and re-raise failures without converting them into success.

We will build the solution in small pieces. Read each prediction question before running the following cell; then compare your prediction with the output and assertions.

### Step 1 — Create an operation that may fail

The endpoint records work and either returns a number or raises `ValueError`. The wrapper must not duplicate this operation. This challenge is about correct *control flow* around `super()`: `try`, `except`, `else`, and `finally` all serve different purposes.

In [17]:
class P05_Engine:
    def __init__(self):
        self.events = []

    def compute(self, number, *, fail=False):
        self.events.append("engine")
        if fail:
            raise ValueError("computation rejected")
        return number * 3

### Step 2 — Write the complete wrapper

For success, log `start → engine → success → cleanup`. For failure, log `start → engine → failure → cleanup`, and preserve the original `ValueError`. Notice `raise` without an argument inside `except`: this keeps the original exception instead of replacing it. `finally` deliberately does **not** return anything, because a return there would suppress a pending exception or overwrite a result.

In [18]:
class P05_LoggedEngine(P05_Engine):
    def compute(self, number, *, fail=False):
        self.events.append("start")
        try:
            result = super().compute(number, fail=fail)
        except Exception:
            self.events.append("failure")
            raise
        else:
            self.events.append("success")
            return result
        finally:
            self.events.append("cleanup")

### Step 3 — Run success and failure independently

Separating instances makes it easy to inspect complete traces without accidental carry-over from a previous request. **Prediction:** Is `cleanup` emitted before or after the `return` takes effect?

In [19]:
successful = P05_LoggedEngine()
print("value:", successful.compute(4))
print("success trace:", successful.events)

failed = P05_LoggedEngine()
try:
    failed.compute(4, fail=True)
except ValueError as exc:
    print("caught:", str(exc))
else:
    raise AssertionError("the failure was accidentally suppressed")
print("failure trace:", failed.events)

value: 12
success trace: ['start', 'engine', 'success', 'cleanup']
caught: computation rejected
failure trace: ['start', 'engine', 'failure', 'cleanup']


### What happened?

`finally` executes as the function exits, including when exiting via `return` or exception. The `else` branch runs only if the `try` body succeeds. The delegated endpoint runs once per call, and the wrapper does not change its successful numeric result or its failure type.

In [20]:
assert successful.events == ["start", "engine", "success", "cleanup"]
assert failed.events == ["start", "engine", "failure", "cleanup"]
print("P05 solved: return values, exception propagation, and cleanup are correct.")

P05 solved: return values, exception propagation, and cleanup are correct.


### Design lesson

A wrapper should document whether it transforms data, catches exceptions, or only observes them. If the intended policy is *observe and re-raise*, use a bare `raise` and avoid `return` from `finally`. More specialized applications may need to handle `BaseException`-derived events (for example cancellation); the dedicated async problem later addresses that case.

## Problem 06 — Override a property safely, including its inherited setter

**Level:** Expert · descriptors and atomic validation  
**Your assignment:** Implement a constrained temperature sensor with a delegated getter, an explicitly delegated base-property setter, and a Fahrenheit view that cannot bypass the constraint.

We will build the solution in small pieces. Read each prediction question before running the following cell; then compare your prediction with the output and assertions.

### Step 1 — Make a validated storage property

The base stores degrees Celsius as a finite float. Validation happens *before* storage. We exclude booleans even though `bool` is a subclass of `int`. Constructor assignment goes through the property, so subclass overrides of this property are also visible during initialization.

In [21]:
class P06_Sensor:
    def __init__(self, celsius):
        self.celsius = celsius

    @property
    def celsius(self):
        return self._celsius

    @celsius.setter
    def celsius(self, value):
        if (isinstance(value, bool) or not isinstance(value, Real)
                or not math.isfinite(float(value))):
            raise ValueError("Celsius must be a finite real number")
        self._celsius = float(value)

### Step 2 — Impose a narrower rule in the child

A comfort sensor only accepts 18–30 °C. `super().celsius` is an appropriate **getter** delegation, but `super().celsius = x` is **not** a way to call the inherited property's setter. We must access the base property descriptor's `.fset` explicitly or use a separate inherited validation method. Here we choose `.fset` to study the descriptor mechanism.

Validate the stricter condition **before** invoking `.fset`; otherwise a failed child validation could leave the base state changed.

In [22]:
class P06_ComfortSensor(P06_Sensor):
    @property
    def celsius(self):
        return super().celsius

    @celsius.setter
    def celsius(self, value):
        if (isinstance(value, bool) or not isinstance(value, Real)
                or not math.isfinite(float(value)) or not 18 <= value <= 30):
            raise ValueError("comfort temperature must be between 18 and 30 °C")
        P06_Sensor.celsius.fset(self, value)

    @property
    def fahrenheit(self):
        return self.celsius * 9 / 5 + 32

    @fahrenheit.setter
    def fahrenheit(self, value):
        if (isinstance(value, bool) or not isinstance(value, Real)
                or not math.isfinite(float(value))):
            raise ValueError("Fahrenheit must be finite")
        self.celsius = (value - 32) * 5 / 9

### Step 3 — Observe dispatch during construction

The inherited `P06_Sensor.__init__()` executes `self.celsius = celsius`. When `self` is a comfort sensor, **the child's setter** runs. The same rule applies when Fahrenheit is assigned: that setter converts units, then dispatches through `self.celsius`, ensuring the constraint is not bypassed.

In [23]:
sensor = P06_ComfortSensor(20)
print("initial Celsius and Fahrenheit:", sensor.celsius, sensor.fahrenheit)
sensor.fahrenheit = 77
print("after Fahrenheit assignment:", sensor.celsius)
assert math.isclose(sensor.celsius, 25.0)
assert math.isclose(sensor.fahrenheit, 77.0)

initial Celsius and Fahrenheit: 20.0 68.0
after Fahrenheit assignment: 25.0


### Step 4 — Check failure atomicity

A failed assignment must leave the old `_celsius` unchanged. Test an out-of-range Celsius value, an out-of-range Fahrenheit value, and `nan`. The preserved-state checks are as important as the error messages.

In [24]:
old = sensor.celsius
for action in (
    lambda: setattr(sensor, "celsius", 40),
    lambda: setattr(sensor, "fahrenheit", 212),
    lambda: setattr(sensor, "celsius", float("nan")),
    lambda: setattr(sensor, "fahrenheit", True),
):
    try:
        action()
    except ValueError:
        pass
    else:
        raise AssertionError("out-of-contract assignment succeeded")
    assert sensor.celsius == old
print("P06 solved: delegated property logic and state preservation verified.")

P06 solved: delegated property logic and state preservation verified.


### Design lesson

A property is a descriptor, not just a method with a funny name. For an overriding read accessor, `super().property` can retrieve the inherited value. For an overriding write accessor, explicitly use the base descriptor's `.fset(self, value)` or redesign the base with a callable validation/write helper. Parent construction through `self.property = ...` may dispatch into child code before child-specific initialization has happened, so avoid relying on not-yet-created child attributes in such setters.

## Problem 07 — Cooperate correctly with `__slots__`

**Level:** Advanced · object layout  
**Your assignment:** Initialize a slotted base and slotted child with `super()`, prove both initializers mutate one object, and explain why a child without slots changes the storage model.

We will build the solution in small pieces. Read each prediction question before running the following cell; then compare your prediction with the output and assertions.

### Step 1 — Write a slotted base

Slots restrict the ordinary instance attribute layout. We will store the object identity in the base and the child as an observable proof that delegation does not manufacture another instance.

In [25]:
class P07_Node:
    __slots__ = ("name", "identity_trace")

    def __init__(self, name):
        if not isinstance(name, str) or not name:
            raise ValueError("node name required")
        self.name = name
        self.identity_trace = [("base", id(self))]

### Step 2 — Extend layout and initialization

The child declares only its **new** slot, `depth`; inherited slots remain available. `super().__init__(name)` prepares the base slots on the original child instance. We append a child identity entry to the inherited list.

In [26]:
class P07_TreeNode(P07_Node):
    __slots__ = ("depth",)

    def __init__(self, name, depth):
        super().__init__(name)
        if not isinstance(depth, int) or isinstance(depth, bool) or depth < 0:
            raise ValueError("depth must be a nonnegative integer")
        self.depth = depth
        self.identity_trace.append(("child", id(self)))


node = P07_TreeNode("root", 0)
print("attributes:", node.name, node.depth)
print("identity trace:", node.identity_trace)

attributes: root 0
identity trace: [('base', 2305015393792), ('child', 2305015393792)]


### Step 3 — Prove the storage contract

**Prediction:** Does a slotted child expose `node.__dict__`? Not if every class in this inheritance chain uses appropriate slots and none supplies an instance dictionary. Try to assign an unlisted attribute and capture the error. Note: `id()` values vary from one run to another, so test their equality rather than hardcoding an address.

In [27]:
assert node.identity_trace == [("base", id(node)), ("child", id(node))]
assert not hasattr(node, "__dict__")
try:
    node.unexpected = 1
except AttributeError:
    pass
else:
    raise AssertionError("slotted layout unexpectedly accepted an attribute")
print("P07 solved: same receiver; no instance __dict__.")

P07 solved: same receiver; no instance __dict__.


### Design lesson

`super()` preserves the receiver even with special instance layouts. A subclass that omits `__slots__` generally gains an instance `__dict__` unless another layout constraint applies. Slot definitions are about storage, not authorization; they are not a substitute for data validation or other application-level invariants.

## Problem 08 — Build a subclass-preserving classmethod factory

**Level:** Expert · binding and alternate constructors  
**Your assignment:** Implement a factory that normalizes input with dynamic class dispatch, extend it with `super()`, and prove that the final object has the requested subclass type.

We will build the solution in small pieces. Read each prediction question before running the following cell; then compare your prediction with the output and assertions.

### Step 1 — Define a reusable base factory

A class method receives `cls`, not `self`. When a subclass calls the inherited factory through `super()`, Python still binds the factory to the subclass. The base factory uses `cls.normalize(raw)` and `cls(name)` so that both normalization and construction can be specialized.

In [28]:
class P08_Record:
    def __init__(self, name):
        self.name = name

    @classmethod
    def normalize(cls, raw):
        if not isinstance(raw, str) or not raw.strip():
            raise ValueError("name must be nonempty text")
        return raw.strip()

    @classmethod
    def from_text(cls, raw):
        return cls(cls.normalize(raw))

### Step 2 — Extend both normalization and factory behavior

The subclass changes names to lowercase and retains the original input for diagnostics. It delegates `normalize()` to the base's implementation; it delegates `from_text()` to the base factory. **Prediction:** Does the `cls(...)` expression in `P08_Record.from_text()` allocate a `P08_Record` or a `P08_TrackedRecord`?

In [29]:
class P08_TrackedRecord(P08_Record):
    def __init__(self, name):
        super().__init__(name)
        self.raw_input = None

    @classmethod
    def normalize(cls, raw):
        return super().normalize(raw).casefold()

    @classmethod
    def from_text(cls, raw):
        instance = super().from_text(raw)
        instance.raw_input = raw
        return instance


tracked = P08_TrackedRecord.from_text("  MiXeD  ")
print(type(tracked).__name__, tracked.name, repr(tracked.raw_input))

P08_TrackedRecord mixed '  MiXeD  '


### What happened?

The factory is still bound to `P08_TrackedRecord`, so its `cls(...)` expression constructs the subclass and invokes its initializer. Its `cls.normalize(...)` dispatches to the subclass, which delegates to the base normalizer and adds case folding. The outer subclass factory then stores the raw input on that same object.

### Step 3 — Prove dispatch and validate bad input

Test the base independently: its name stays in the original case. Test the subclass: it is constructed at the correct type and carries the original text. The expected `ValueError` is captured so the rest of the notebook can run.

In [30]:
base_record = P08_Record.from_text("  MiXeD  ")
assert type(base_record) is P08_Record
assert base_record.name == "MiXeD"
assert type(tracked) is P08_TrackedRecord
assert (tracked.name, tracked.raw_input) == ("mixed", "  MiXeD  ")
try:
    P08_TrackedRecord.from_text("   ")
except ValueError:
    pass
else:
    raise AssertionError("blank input should be rejected")
print("P08 solved: class binding survives delegation.")

P08 solved: class binding survives delegation.


### Design lesson

Class-method delegation retains dynamic class binding, just as instance-method delegation retains dynamic object binding. Factory authors should intentionally decide whether to call `cls(...)` (subclass-preserving) or a named concrete class (type-fixed). The correct choice depends on the constructor contract of subclasses; this example keeps that contract compatible.

## Problem 09 — Delegate construction of an immutable tuple subtype

**Level:** Expert · `__new__`  
**Your assignment:** Validate and allocate a tuple subclass using `super().__new__`, then add a labeled subclass without trying to change immutable tuple contents inside `__init__`.

We will build the solution in small pieces. Read each prediction question before running the following cell; then compare your prediction with the output and assertions.

### Step 1 — Why `__init__` is not enough

For an immutable built-in such as `tuple`, the tuple elements are established in `__new__`, before `__init__` runs. The usual `self.start = ...` approach would add an attribute, not change the underlying tuple. We implement a two-int interval stored as `(start, end)` with `start <= end`.

In [31]:
class P09_Interval(tuple):
    def __new__(cls, start, end):
        if (type(start) is not int or type(end) is not int or start > end):
            raise ValueError("interval needs two ordered integers")
        return super().__new__(cls, (start, end))

    def __init__(self, start, end):
        super().__init__()

    @property
    def start(self):
        return self[0]

    @property
    def end(self):
        return self[1]

    @property
    def length(self):
        return self.end - self.start

### Step 2 — Specialize allocation and preserve the tuple payload

The labeled subclass validates its new `tag` **before** delegating allocation, then delegates the tuple construction to the inherited `__new__`. Its `__init__` calls the parent initializer and stores metadata. Ordinary tuple elements stay immutable; metadata is a separate attribute.

In [32]:
class P09_TaggedInterval(P09_Interval):
    def __new__(cls, start, end, tag):
        if not isinstance(tag, str) or not tag.strip():
            raise ValueError("tag must be nonempty")
        return super().__new__(cls, start, end)

    def __init__(self, start, end, tag):
        super().__init__(start, end)
        self.tag = tag


interval = P09_TaggedInterval(3, 8, "window")
print("tuple payload:", tuple(interval))
print("metadata:", interval.tag, "length:", interval.length)

tuple payload: (3, 8)
metadata: window length: 5


### Step 3 — Test the real immutability boundary

A tuple subclass's elements cannot be reassigned, but its extra attributes may remain mutable if it has an instance dictionary. Distinguish these two claims rather than calling every aspect of the object “immutable.”

In [33]:
assert isinstance(interval, tuple)
assert isinstance(interval, P09_Interval)
assert (interval.start, interval.end, interval.length) == (3, 8, 5)
try:
    interval[0] = 4
except TypeError:
    pass
else:
    raise AssertionError("tuple contents unexpectedly mutated")
interval.tag = "renamed"
assert interval.tag == "renamed"
for args in ((8, 3, "bad"), (True, 3, "bad"), (1, 3, "  ")):
    try:
        P09_TaggedInterval(*args)
    except ValueError:
        pass
    else:
        raise AssertionError(f"invalid interval allowed: {args!r}")
print("P09 solved: allocation and metadata have distinct responsibilities.")

P09 solved: allocation and metadata have distinct responsibilities.


### Design lesson

`super().__new__(cls, ...)` is relevant whenever allocation or immutable payload creation happens before initialization. An overriding `__init__` alone cannot repair an incorrectly constructed immutable value. This pattern is specialized: use regular classes or dataclasses for most mutable domain objects, and keep allocation logic minimal and well tested.

## Problem 10 — Cooperate in context-manager entry and exit

**Level:** Expert · protocols  
**Your assignment:** Build a context manager with a delegated `__enter__` and `__exit__`, prove cleanup runs on exceptions, and make exception suppression an explicit policy rather than an accident.

We will build the solution in small pieces. Read each prediction question before running the following cell; then compare your prediction with the output and assertions.

### Step 1 — Create a base resource

Python's `with` protocol invokes `__enter__()` and ultimately `__exit__(exc_type, exc, traceback)`. Returning a truthy value from `__exit__` suppresses an exception raised within the block; returning false leaves it propagating. Our base **never suppresses**.

In [34]:
class P10_Resource:
    def __init__(self):
        self.events = []
        self.is_open = False

    def __enter__(self):
        self.events.append("base-enter")
        self.is_open = True
        return self

    def __exit__(self, exc_type, exc, tb):
        self.events.append("base-exit")
        self.is_open = False
        return False

### Step 2 — Add a tracing layer without losing the returned resource

`__enter__` must return the resource from `super().__enter__()`, not accidentally return `None`. `__exit__` must forward **all three exception arguments** and return the delegated suppression decision. Our tracing layer logs its own exit before releasing the base resource.

In [35]:
class P10_Trace(P10_Resource):
    def __enter__(self):
        resource = super().__enter__()
        self.events.append("trace-enter")
        return resource

    def __exit__(self, exc_type, exc, tb):
        self.events.append("trace-exit")
        return super().__exit__(exc_type, exc, tb)


resource = P10_Trace()
with resource as bound:
    assert bound is resource
    assert bound.is_open
    bound.events.append("work")
print(resource.events, resource.is_open)

['base-enter', 'trace-enter', 'work', 'trace-exit', 'base-exit'] False


### Step 3 — Inspect an exceptional exit

**Prediction:** If the block raises `RuntimeError`, will the base cleanup run? Will the exception propagate? Catch the exception *outside* the `with` so that we can inspect the complete event list.

In [36]:
broken_resource = P10_Trace()
try:
    with broken_resource:
        broken_resource.events.append("raise")
        raise RuntimeError("unexpected work failure")
except RuntimeError:
    broken_resource.events.append("caught-outside")
else:
    raise AssertionError("RuntimeError was unexpectedly suppressed")
print(broken_resource.events)

['base-enter', 'trace-enter', 'raise', 'trace-exit', 'base-exit', 'caught-outside']


### What happened?

When `with` exits, tracing delegates to base `__exit__`, so `is_open` becomes `False` even for errors. The base returns `False`, and tracing returns that result, so the `RuntimeError` remains visible to the outer exception handler.

### Step 4 — Introduce narrowly targeted suppression

Now create a subclass that *deliberately* suppresses only `KeyError`, while still performing all parent cleanup. This is a policy decision rather than a side effect of forgetting the parent's return value. A different exception such as `ValueError` must still propagate.

In [37]:
class P10_SuppressMissing(P10_Trace):
    def __exit__(self, exc_type, exc, tb):
        parent_suppressed = super().__exit__(exc_type, exc, tb)
        return parent_suppressed or (exc_type is not None and issubclass(exc_type, KeyError))


quiet = P10_SuppressMissing()
with quiet:
    raise KeyError("missing")  # intentional; narrowly suppressed by policy

assert quiet.events == ["base-enter", "trace-enter", "trace-exit", "base-exit"]
assert not quiet.is_open
assert resource.events == ["base-enter", "trace-enter", "work", "trace-exit", "base-exit"]
assert broken_resource.events == [
    "base-enter", "trace-enter", "raise", "trace-exit", "base-exit", "caught-outside"
]
print("P10 solved: lifecycle and suppression policy verified.")

P10 solved: lifecycle and suppression policy verified.


### Design lesson

Lifecycle protocols have return-value contracts. Forwarding with `super()` is only half the solution: preserve the return from `__enter__`, pass exception details to `__exit__`, and decide explicitly whether to propagate or suppress. Production cleanup code may also need to handle errors *during cleanup*; that is a separate policy not hidden in this exercise.

## Problem 11 — Await `super()` and propagate cancellation correctly

**Level:** Expert · asynchronous methods  
**Your assignment:** Write an async logging mixin that delegates a coroutine, preserves its result, records errors, and does not accidentally swallow task cancellation.

We will build the solution in small pieces. Read each prediction question before running the following cell; then compare your prediction with the output and assertions.

### Step 1 — Build two independent asynchronous endpoints

One service returns after an event-loop yield, optionally raising a regular exception. The second waits on an unset `asyncio.Event`, allowing a deterministic cancellation test. We will use **top-level `await`**, which Jupyter notebooks support; calling `asyncio.run()` inside a running notebook event loop is the wrong pattern.

In [38]:
import asyncio


class P11_QuickEndpoint:
    def __init__(self):
        self.events = []

    async def fetch(self, *, fail=False):
        self.events.append("endpoint")
        await asyncio.sleep(0)
        if fail:
            raise RuntimeError("endpoint failed")
        return "data"


class P11_WaitingEndpoint:
    def __init__(self):
        self.events = []
        self.gate = asyncio.Event()

    async def fetch(self, *, fail=False):
        self.events.append("waiting")
        await self.gate.wait()
        return "released"

### Step 2 — Implement a genuinely reusable mixin

The mixin does not name either endpoint. It calls `await super().fetch(...)`, catches cancellation separately, and records final cleanup. In modern Python, `asyncio.CancelledError` derives from `BaseException`, not `Exception`; catching only ordinary exceptions does not record cancellations. Never return from `finally` or silently turn cancellation into a normal value.

In [39]:
class P11_LoggingMixin:
    async def fetch(self, *, fail=False):
        self.events.append("start")
        try:
            result = await super().fetch(fail=fail)
        except asyncio.CancelledError:
            self.events.append("cancelled")
            raise
        except Exception:
            self.events.append("error")
            raise
        else:
            self.events.append("success")
            return result + "-wrapped"
        finally:
            self.events.append("cleanup")


class P11_QuickService(P11_LoggingMixin, P11_QuickEndpoint):
    pass


class P11_WaitingService(P11_LoggingMixin, P11_WaitingEndpoint):
    pass

### Step 3 — Test the successful and failed calls

A cell in a Jupyter notebook may directly `await` a coroutine. We also test that neither the exception type nor the single endpoint call is lost when the mixin delegates.

In [40]:
quick = P11_QuickService()
received = await quick.fetch()
assert received == "data-wrapped"
assert quick.events == ["start", "endpoint", "success", "cleanup"]

failed_async = P11_QuickService()
try:
    await failed_async.fetch(fail=True)
except RuntimeError as exc:
    assert "endpoint failed" in str(exc)
else:
    raise AssertionError("endpoint failure disappeared")
assert failed_async.events == ["start", "endpoint", "error", "cleanup"]
print("success and error traces verified")

success and error traces verified


### Step 4 — Test cancellation without sleeps or races

Create a task, yield control until the waiting endpoint has entered its wait, then cancel the task and await it inside an expected-exception handler. This checks both propagation and cleanup. The small handoff loop uses `asyncio.sleep(0)` only to let the scheduled task execute; it is **not** a time-based assertion.

In [41]:
waiting = P11_WaitingService()
task = asyncio.create_task(waiting.fetch())
for _ in range(20):
    if "waiting" in waiting.events:
        break
    await asyncio.sleep(0)
else:
    task.cancel()
    raise AssertionError("task never reached the waiting endpoint")

task.cancel()
try:
    await task
except asyncio.CancelledError:
    pass
else:
    raise AssertionError("cancellation was swallowed")

assert waiting.events == ["start", "waiting", "cancelled", "cleanup"]
print("P11 solved: cancellation propagated; cleanup completed.")

P11 solved: cancellation propagated; cleanup completed.


### Design lesson

A coroutine returned by an async method is not its final result; delegation requires `await super().method()`. Preserve the cancellation semantics of the surrounding application. If a coroutine catches `BaseException` indiscriminately, it can suppress cancellation and other control-flow exceptions. This example intentionally catches cancellation only to log and immediately re-raise it.

## Problem 12 — Use an abstract method with a cooperative default body

**Level:** Expert · ABCs and cooperative implementation  
**Your assignment:** Create an abstract rendering protocol whose abstract method provides reusable code, then implement nested renderers through `super()` and prove that the final class is instantiable.

We will build the solution in small pieces. Read each prediction question before running the following cell; then compare your prediction with the output and assertions.

### Step 1 — Understand the less obvious ABC feature

An `@abstractmethod` is allowed to contain an implementation. It prevents direct instantiation until some subclass supplies a concrete override, but that override can still call the abstract method's existing implementation through `super()`. We use that method as the terminal content provider.

In [42]:
from abc import ABC, abstractmethod


class P12_Renderable(ABC):
    @abstractmethod
    def render(self):
        return "content"


try:
    P12_Renderable()
except TypeError:
    print("Abstract base cannot be instantiated: expected")
else:
    raise AssertionError("abstract base unexpectedly instantiable")

Abstract base cannot be instantiated: expected


### Step 2 — Create independently composed renderers

Each renderer overrides `render()` and wraps the result of `super().render()`. On a combined document, the next method is determined by the combined MRO, not by the wrapper's immediate declared base. The leading implementation adds a terminator.

In [43]:
class P12_Italic(P12_Renderable):
    def render(self):
        return "<i>" + super().render() + "</i>"


class P12_Bold(P12_Renderable):
    def render(self):
        return "<b>" + super().render() + "</b>"


class P12_Document(P12_Italic, P12_Bold):
    def render(self):
        return super().render() + "!"


show_mro(P12_Document)
document = P12_Document()
print(document.render())

P12_Document -> P12_Italic -> P12_Bold -> P12_Renderable -> ABC -> object
<i><b>content</b></i>!


### What happened?

The abstract endpoint contributes its normal default body after all wrappers, because a concrete method override exists in the hierarchy. `P12_Document` is instantiable; it is not necessary for every class to replace the entire body of an abstract method. The wrappers remain cooperative, so both markup layers are present.

In [44]:
assert not P12_Document.__abstractmethods__
assert document.render() == "<i><b>content</b></i>!"
assert P12_Document.__mro__[:4] == (
    P12_Document, P12_Italic, P12_Bold, P12_Renderable
)
print("P12 solved: abstract contract and cooperative endpoint both work.")

P12 solved: abstract contract and cooperative endpoint both work.


### Design lesson

An abstract method can define a useful endpoint for a `super()` chain. However, do not assume every abstract method is automatically a cooperative endpoint: if its body is `pass`, returns `None`, or raises `NotImplementedError`, delegating into that body may be inappropriate. Inspect the contract and make the terminal implementation explicit.

## Problem 13 — Capstone: cooperative, collision-safe serialization

**Level:** Expert · integrated design  
**Your assignment:** Combine independent metadata layers, a strict constructor chain, return-value composition, collision detection, stable object identity, and tests for invalid inputs and accidental key overwrites.

We will build the solution in small pieces. Read each prediction question before running the following cell; then compare your prediction with the output and assertions.

### Step 1 — State the problem before implementing it

We want a `Document` with `record_id`, `owner`, `created_at`, and `title`. Each class owns one field and contributes exactly one serialization key to `to_dict()`. The root owns `record_id` and returns a **new dictionary on every call**; mixins must not mutate some shared class-level object. If a downstream layer already supplied a key that a mixin owns, that mixin must raise instead of silently overwriting it.

The cooperative MRO will be `Document → OwnerMixin → TimeMixin → Root → object`. Constructor arguments are keyword-only, and the endpoint rejects unknown options.

### Step 2 — Implement the strict root

The root has two roles: enforce a predictable `__init__` endpoint and supply the `to_dict()` endpoint. It validates `record_id` before modifying the instance. We deliberately make `to_dict()` return a fresh dictionary so repeated calls do not reuse mutable state.

In [45]:
class P13_Root:
    def __init__(self, *, record_id, **kwargs):
        if kwargs:
            raise TypeError(f"unknown options: {sorted(kwargs)}")
        if not isinstance(record_id, str) or not record_id.strip():
            raise ValueError("record_id must be nonempty")
        self.record_id = record_id
        self.init_trace = ["root"]

    def to_dict(self):
        return {"record_id": self.record_id}

### Step 3 — Implement an independent time layer

The time mixin inherits no particular application base: its only assumption is that the next class in the MRO offers compatible `__init__` and `to_dict` methods. Its constructor consumes `created_at`, forwards the remaining keywords, and appends its trace after downstream initialization. Its serializer checks for a key collision before returning an expanded **new** dictionary.

In [46]:
class P13_TimeMixin:
    def __init__(self, *, created_at, **kwargs):
        if not isinstance(created_at, str) or not created_at.strip():
            raise ValueError("created_at must be nonempty text")
        super().__init__(**kwargs)
        self.created_at = created_at
        self.init_trace.append("time")

    def to_dict(self):
        data = super().to_dict()
        if "created_at" in data:
            raise KeyError("created_at collision")
        return {**data, "created_at": self.created_at}

### Step 4 — Implement an independent owner layer

Owner validation and serialization follow the same protocol, but this layer owns a different field. By avoiding any reference to the concrete document class, we keep it reusable and ensure the next implementation is determined by MRO.

In [47]:
class P13_OwnerMixin:
    def __init__(self, *, owner, **kwargs):
        if not isinstance(owner, str) or not owner.strip():
            raise ValueError("owner must be nonempty text")
        super().__init__(**kwargs)
        self.owner = owner
        self.init_trace.append("owner")

    def to_dict(self):
        data = super().to_dict()
        if "owner" in data:
            raise KeyError("owner collision")
        return {**data, "owner": self.owner}

### Step 5 — Compose the concrete document

`P13_Document` consumes its own title, delegates everything else to the cooperative chain, and contributes `title` on the way back. **Prediction:** What is the order of `init_trace`? In what order are the dictionary keys inserted? The two orders are governed by different moments: post-constructor statements and nested return values.

In [48]:
class P13_Document(P13_OwnerMixin, P13_TimeMixin, P13_Root):
    def __init__(self, *, title, **kwargs):
        if not isinstance(title, str) or not title.strip():
            raise ValueError("title must be nonempty text")
        super().__init__(**kwargs)
        self.title = title
        self.init_trace.append("document")

    def to_dict(self):
        data = super().to_dict()
        if "title" in data:
            raise KeyError("title collision")
        return {**data, "title": self.title}


show_mro(P13_Document)
capstone = P13_Document(
    title="Release notes", owner="Ari", created_at="2026-09-20", record_id="DOC-7"
)
print("constructor trace:", capstone.init_trace)
print("serialized:", capstone.to_dict())

P13_Document -> P13_OwnerMixin -> P13_TimeMixin -> P13_Root -> object
constructor trace: ['root', 'time', 'owner', 'document']
serialized: {'record_id': 'DOC-7', 'created_at': '2026-09-20', 'owner': 'Ari', 'title': 'Release notes'}


### What happened?

Construction descends into the root and finishes outwards, giving `root → time → owner → document`. Serialization descends into the root and then combines return values outwards; the result contains the four keys in `record_id`, `created_at`, `owner`, `title` insertion order. More importantly, each class contributes only its own portion and each call returns a fresh dictionary.

### Step 6 — Verify data, independence, and strict input handling

These assertions check more than a representative output: all keys, exact constructor participation, independent returned dictionaries, and rejection of an unknown argument. Mutating one returned dictionary must not mutate the document or another returned dictionary.

In [49]:
expected = {
    "record_id": "DOC-7",
    "created_at": "2026-09-20",
    "owner": "Ari",
    "title": "Release notes",
}
assert capstone.init_trace == ["root", "time", "owner", "document"]
first = capstone.to_dict()
second = capstone.to_dict()
assert first == second == expected
assert first is not second
first["owner"] = "modified-output-only"
assert capstone.owner == "Ari"
assert second == expected
try:
    P13_Document(title="X", owner="Y", created_at="today", record_id="R", typo=1)
except TypeError as exc:
    assert "typo" in str(exc)
else:
    raise AssertionError("typos must not be ignored")
print("strict inputs and independent output dictionaries verified")

strict inputs and independent output dictionaries verified


### Step 7 — Prove the collision check is real

A malicious or buggy downstream implementation might already claim an `owner` key. Compose the owner mixin with a different root that intentionally returns this conflicting key. The correct behavior is a `KeyError`, not silently replacing `intruder` with our owner value. This test also shows the mixin did not hardcode the identity of its successor.

In [50]:
class P13_ConflictingRoot(P13_Root):
    def to_dict(self):
        return {**super().to_dict(), "owner": "intruder"}


class P13_CollisionDocument(P13_OwnerMixin, P13_ConflictingRoot):
    pass


collision = P13_CollisionDocument(owner="genuine", record_id="DOC-8")
try:
    collision.to_dict()
except KeyError as exc:
    assert "owner collision" in str(exc)
else:
    raise AssertionError("duplicate serializer key was silently overwritten")
print("P13 solved: cooperative serialization, no silent key collisions.")

P13 solved: cooperative serialization, no silent key collisions.


### Capstone design review

**Why this architecture works:** (1) every initializer consumes its own keyword; (2) the root rejects leftovers; (3) every override calls `super()` once; (4) return values are preserved and explicitly extended; (5) dictionary output is fresh; (6) collisions fail loudly; (7) testing covers happy paths, invalid inputs, and an adversarial composition.

**Trade-off:** Cooperative mixins rely on a shared method/argument protocol, so they deserve integration tests for each intended MRO. For a simple fixed schema, plain composition or a dataclass might be easier to maintain; use inheritance here because the *purpose of this lesson* is to understand how reusable delegation behaves under composition.

---

# Final review: reason from the receiver, the MRO, and the method contract

The uploaded lesson established three core observations: an overriding method can delegate to a version higher in the hierarchy; the delegated method still receives the original object; and a parent method calling `self.some_method()` can dynamically select a child's override. Our new examples apply those foundations to **different problems and protocols**.

For every unfamiliar `super()` expression, answer these questions in order:

1. **What is the runtime receiver?** Identify the actual object, or the bound class for a class method.
2. **Which method contains the `super()` expression?** Lookup continues *after that class* in the receiver's MRO.
3. **What must be forwarded?** Preserve positional/keyword arguments and protocol-specific exception information.
4. **What must be returned?** For a method, constructor, context manager, coroutine, property, or immutable allocation, the return contract differs.
5. **What must happen on error?** Decide whether to re-raise, clean up, roll back, or intentionally suppress; do not let accidental control flow choose.
6. **What should your tests assert?** Verify call order, result, receiver identity, negative cases, and invariants—not only that no exception was raised.

**Best-practice limit:** Not every hierarchy benefits from `super()`; prefer straightforward attribute access when no override needs delegation, and consider composition when cooperative class protocols become hard to understand.

In [51]:
# A quick, independent final smoke test across the entire tutorial.
assert P01_Combined().route() == "L(R(core))"
assert P02_Safe().run() == "safe-run:base-run:safe-base-prepare"
assert P03_AuditedLedger().post(1) == 1.0
assert P04_Job(job_id="j", name="n", started_at=0).steps == ["root", "timed", "named", "job"]
assert P05_LoggedEngine().compute(2) == 6
assert P06_ComfortSensor(22).celsius == 22.0
assert P07_TreeNode("n", 1).depth == 1
assert P08_TrackedRecord.from_text(" X ").name == "x"
assert P09_TaggedInterval(1, 2, "t").length == 1
assert P10_Trace().is_open is False
assert P12_Document().render() == "<i><b>content</b></i>!"
assert capstone.to_dict() == expected
print("Final smoke test: all representative synchronous examples passed.")

Final smoke test: all representative synchronous examples passed.


## Further practice (with solution hints)

These short transfer exercises are intentionally left for independent extension **after** the fully worked problems. They are not prerequisites for any tests above.

- **Reverse wrapper order:** Change `P01_Combined(P01_Left, P01_Right)` to `(P01_Right, P01_Left)`. **Hint / expected conclusion:** the wrapper nesting and event order reverse; the core still runs once.
- **Add a third ledger hook:** Extend `P03_AuditedLedger` with a subclass that increments a `success_count` in `_record()`. **Hint:** delegate first, then increment; invalid `post()` calls never reach `_record()`.
- **Strengthen context handling:** Change `P10_SuppressMissing` to suppress only `KeyError` raised *directly* (not its subclasses). **Hint:** compare with `exc_type is KeyError` instead of `issubclass`.
- **Extend the serializer:** Add a `P13_CategoryMixin` with its own `category` keyword and `to_dict()` key. **Hint:** copy the established constructor/serialization protocol and test with different base orders.

**Completion criterion:** Every executable cell runs successfully from a clean kernel, including the async `await` cells; no expected failure is left as an uncaught exception.